In [ ]:

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Lambda

In [ ]:
# Load pretrained ResNet50 model
# include_top=False → remove classification layer
# input_shape → our custom image size

base_network = ResNet50(
    weights="imagenet",
    input_shape=(200,200,3),
    include_top=False
)

# Freeze most of the layers so pretrained weights stay same
# Only last few layers will train
for layer in base_network.layers[:-3]:
    layer.trainable = False

# Convert CNN feature map to vector
pool = GlobalAveragePooling2D()(base_network.output)

# Dense layer to learn embedding features
dense1 = Dense(128, activation='tanh')(pool)

# Final embedding vector (128 dimension)
output = Dense(128)(dense1)

# Create embedding model
# This model converts image → feature vector
embedding = Model(base_network.input, output, name="Embedding")

In [ ]:
# This function calculates distance between two feature vectors
def euclidean_distance(vects):
    x,y =vects
    # (x - y)^2
    sum_square =K.sum(K.square(x - y),axis=1,keepdims=True)
    # sqrt(sum of squares)
    # epsilon added to avoid sqrt(0)
    return K.sqrt(K.maximum(sum_square,K.epsilon()))

In [ ]:
# First image input
img_1 = Input(shape=(200,200,3), name="input_1")

# Second image input
img_2 = Input(shape=(200,200,3), name="input_2")

# Pass both images through SAME embedding network
# This is the key idea of Siamese networks

feature_vector_1 = embedding(img_1)
feature_vector_2 = embedding(img_2)

# Calculate distance between embeddings
distance = Lambda(euclidean_distance)([feature_vector_1, feature_vector_2])

# Final Siamese model
# Input → two images
# Output → distance between them
siamese_model = Model(inputs=[img_1, img_2], outputs=distance)

In [9]:
def contrastive_loss(y_true,y_pred,margin=1):
    square_distance_similar =K.square(y_pred)
    square_distance_non_similar = K.square(K.maximum(margin - y_pred,0))
    loss = K.mean(y_true * square_distance_similar + (1-y_true)*(square_distance_non_similar))

In [10]:
siamese_model.compile(
    loss=contrastive_loss,
    optimizer=Adam(0.0001)
)

In [13]:

siamese_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_1             │ (None, 200, 200,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_2             │ (None, 200, 200,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Embedding           │ (None, 128)       │ 23,866,496 │ input_1[0][0],    │
│ (Functional)        │                   │            │ input_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1)         │          0 │ Embedding[0][0],  │
│                     │                   │            │ Embedding[1][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,866,496 (91.04 MB)

 Trainable params: 282,880 (1.08 MB)

 Non-trainable params: 23,583,616 (89.96 MB)